In [2]:
import pandas as pd
import numpy as np


In [3]:
# 读取合并后的数据
df = pd.read_csv('out_data/allmonth.csv',
                 dtype={'product_id': 'int64', 'category_id': 'int64', 'user_id': 'int64'})

# 1.删除多余字段
df = df.drop(columns=['category_code', 'user_session'])

# 2.过滤事件类型
df = df[df['event_type'].isin(['view', 'cart', 'purchase'])]

# 3.删掉核心字段缺失的值
before = len(df)
df = df.dropna(subset=['user_id', 'product_id', 'event_time', 'event_type'])
print(f'删除核心字段缺失 {before - len(df):,} 条，剩余 {len(df):,}')

# 4.删掉异常价格
before = len(df)
df = df[df['price'] > 0]
print(f'删除异常价格 {before - len(df):,} 条，剩余 {len(df):,}')

# 5.填充 brand 缺失值
brand_mode = (
    df.dropna(subset=['brand'])
    .groupby('category_id')['brand']
    .agg(lambda s: s.mode().iloc[0])
)
df['brand'] = df['brand'].fillna(df['category_id'].map(brand_mode))
df['brand'] = df['brand'].fillna('未知')

# 6.去重
before = len(df)
df = df.drop_duplicates()
print(f'去除重复 {before - len(df):,} 条，剩余 {len(df):,}')

# 7.类型处理
# df['event_time'] = pd.to_datetime(df['event_time'])
# df['product_id'] = df['product_id'].astype(str)
# df['category_id'] = df['category_id'].astype(str)
# df['user_id'] = df['user_id'].astype(str)

# 8.保存数据
df.to_csv('out_data/allmonth_clean.csv', index=False, encoding='utf-8')
print(f'清洗完成，最终 {len(df):,} 条 -> out_data/allmonth_clean.csv')


删除核心字段缺失 0 条，剩余 16,713,161
删除异常价格 79,989 条，剩余 16,633,172
去除重复 113,379 条，剩余 16,519,793
清洗完成，最终 16,519,793 条 -> out_data/allmonth_clean.csv
